In [50]:
import pandas as pd
import kagglehub
import os
import random

Importar o dataset a ser utilizado

In [51]:
os.environ["KAGGLEHUB_CACHE"] = "."
# Download latest version
path = kagglehub.dataset_download("dhivyeshrk/diseases-and-symptoms-dataset")

print("Path to dataset files:", path)
files = os.listdir(path)
print(files)

Path to dataset files: .\datasets\dhivyeshrk\diseases-and-symptoms-dataset\versions\1
['Final_Augmented_dataset_Diseases_and_Symptoms.csv']


In [52]:
path = os.path.join(os.getcwd(), "datasets", "dhivyeshrk", "diseases-and-symptoms-dataset", "versions", "1", "Final_Augmented_dataset_Diseases_and_Symptoms.csv")
save_path = "../bronze/Bronze.csv"
df = pd.read_csv(path)
df.head()
df.to_csv(save_path, index=False)

Agrupar os sintomas condizentes a cada doença

In [53]:
col_doenca = 'diseases'

sintomas_cols = df.columns.drop(col_doenca)

def extrair_sintomas_linha(row):
    return [col for col in sintomas_cols if row[col] == 1]

# cria lista de sintomas por linha
df['sintomas'] = df.apply(extrair_sintomas_linha, axis=1)

C:\Users\victo\AppData\Local\Temp\ipykernel_23388\2002239140.py:9: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df['sintomas'] = df.apply(extrair_sintomas_linha, axis=1)


In [54]:
df.head()

,diseases,anxiety and nervousness,depression,shortness of breath,depressive or psychotic symptoms,sharp chest pain,dizziness,insomnia,abnormal involuntary movements,chest tightness,...,problems with orgasm,nose deformity,lump over jaw,sore in nose,hip weakness,back swelling,ankle stiffness or tightness,ankle weakness,neck weakness,sintomas
0,panic disorder,1,0,1,1,0,0,0,0,1,...,0,0,0,0,0,0,0,0,0,"[anxiety and nervousness, shortness of breath,..."
1,panic disorder,0,0,1,1,0,1,1,0,0,...,0,0,0,0,0,0,0,0,0,"[shortness of breath, depressive or psychotic ..."
2,panic disorder,1,1,1,1,0,1,1,0,0,...,0,0,0,0,0,0,0,0,0,"[anxiety and nervousness, depression, shortnes..."
3,panic disorder,1,0,0,1,0,1,1,1,0,...,0,0,0,0,0,0,0,0,0,"[anxiety and nervousness, depressive or psycho..."
4,panic disorder,1,1,0,0,0,0,1,1,1,...,0,0,0,0,0,0,0,0,0,"[anxiety and nervousness, depression, insomnia..."


Organizar para o silver um dataset mais bem estruturado

In [55]:
import re

# Remove colunas duplicadas do pandas (ex: regurgitation.1)
sintomas_cols_limpos = [col for col in sintomas_cols if not re.search(r'\.\d+$', col)]

def extrair_sintomas_linha_limpo(row):
    return [col for col in sintomas_cols_limpos if row[col] == 1]

df['sintomas'] = df.apply(extrair_sintomas_linha_limpo, axis=1)

df_silver = df[[col_doenca, 'sintomas']].copy()
df_silver.head(10)

,diseases,sintomas
0,panic disorder,"[anxiety and nervousness, shortness of breath,..."
1,panic disorder,"[shortness of breath, depressive or psychotic ..."
2,panic disorder,"[anxiety and nervousness, depression, shortnes..."
3,panic disorder,"[anxiety and nervousness, depressive or psycho..."
4,panic disorder,"[anxiety and nervousness, depression, insomnia..."
5,panic disorder,"[shortness of breath, depressive or psychotic ..."
6,panic disorder,[anxiety and nervousness]
7,panic disorder,"[depressive or psychotic symptoms, insomnia, a..."
8,panic disorder,"[anxiety and nervousness, depressive or psycho..."
9,panic disorder,"[anxiety and nervousness, depression, shortnes..."


In [56]:
df_silver[col_doenca] = df_silver[col_doenca].str.lower().str.strip()

df_silver['sintomas'] = df_silver['sintomas'].apply(
    lambda lista: [s.replace('_', ' ').lower() for s in lista]
)

df_silver.head(10)

df_silver.to_csv("../silver/Silver.csv", index=False)

In [57]:
df_silver['sintomas'] = df_silver['sintomas'].apply(
    lambda x: ast.literal_eval(x) if isinstance(x, str) else x
)
contagem = df_silver.groupby('diseases').size().reset_index(name='qtd_linhas')

print(f"Total de linhas no Silver: {len(df_silver)}")
print(f"Total de doenças únicas: {df_silver['diseases'].nunique()}")
print(f"\nDistribuição de linhas por doença:")
print(contagem['qtd_linhas'].describe())
print(f"\nDoenças com menos de 20 linhas: {(contagem['qtd_linhas'] < 20).sum()}")
print(f"Doenças com 20-100 linhas: {((contagem['qtd_linhas'] >= 20) & (contagem['qtd_linhas'] < 100)).sum()}")
print(f"Doenças com 100+ linhas: {(contagem['qtd_linhas'] >= 100).sum()}")

Total de linhas no Silver: 246945
Total de doenças únicas: 773

Distribuição de linhas por doença:
count     773.000000
mean      319.463131
std       351.691549
min         1.000000
25%        30.000000
50%       168.000000
75%       505.000000
max      1219.000000
Name: qtd_linhas, dtype: float64

Doenças com menos de 20 linhas: 166
Doenças com 20-100 linhas: 164
Doenças com 100+ linhas: 443


Agora no gold, transformar e diversificar as informações em textos para interpretabilidade do RAG

In [58]:
random.seed(42)

def gerar_amostras(disease, sintomas, n_amostras=100):
    templates = [
        lambda s: f"{disease.capitalize()} may cause symptoms such as {', '.join(s)}.",
        lambda s: f"Common symptoms of {disease} include {', '.join(s)}.",
        lambda s: f"{disease.capitalize()} is often linked to symptoms like {', '.join(s)}.",
        lambda s: f"Patients with {disease} frequently experience {', '.join(s)}.",
        lambda s: f"A case of {disease} can be associated with {', '.join(s)}.",
        lambda s: f"Patient presents with {', '.join(s)}, which may indicate {disease}.",
        lambda s: f"Clinical presentation includes {', '.join(s)}, suggesting possible {disease}.",
        lambda s: f"Reported symptoms: {', '.join(s)}. Possible condition: {disease}.",
        lambda s: f"Upon evaluation, patient shows {', '.join(s)}, consistent with {disease}.",
        lambda s: f"I have been experiencing {', '.join(s)}. Could this be {disease}?",
        lambda s: f"I am feeling {', '.join(s)}. These symptoms might be related to {disease}.",
        lambda s: f"My symptoms include {', '.join(s)}, which could be associated with {disease}.",
        lambda s: f"What condition causes {', '.join(s)}? One possibility is {disease}.",
        lambda s: f"Is {disease} related to symptoms like {', '.join(s)}?",
        lambda s: f"Can {', '.join(s)} be signs of {disease}?",
    ]

    sinonimos = {
        "fever": ["high temperature", "elevated fever", "febrile state"],
        "headache": ["head pain", "cephalgia", "head discomfort"],
        "fatigue": ["tiredness", "exhaustion", "lack of energy"],
        "cough": ["coughing", "persistent cough", "dry cough"],
        "nausea": ["feeling nauseous", "upset stomach", "queasiness"],
        "vomiting": ["throwing up", "emesis", "vomiting episodes"],
        "diarrhea": ["loose stools", "diarrhea episodes", "frequent bowel movements"],
        "pain": ["discomfort", "aching", "soreness"],
        "rash": ["skin rash", "skin irritation", "cutaneous eruption"],
        "chills": ["shivering", "feeling cold", "rigors"],
    }

    def aplicar_sinonimos(sintoma):
        for chave, variantes in sinonimos.items():
            if chave in sintoma:
                return random.choice([sintoma] + variantes)
        return sintoma

    amostras = set()
    tentativas = 0
    max_tentativas = n_amostras * 100

    while len(amostras) < n_amostras and tentativas < max_tentativas:
        tentativas += 1

        k = random.randint(2, len(sintomas)) if len(sintomas) > 2 else len(sintomas)
        
        # Embaralha a ordem dos sintomas no subset
        subset = random.sample(sintomas, k)
        random.shuffle(subset)

        subset_variado = [aplicar_sinonimos(s) for s in subset]

        template = random.choice(templates)
        texto = template(subset_variado)
        amostras.add(texto)

    return list(amostras)


# Regera o Gold enriquecido
registros = []

for disease, grupo in df_silver.groupby('diseases'):
    todos_sintomas = list({s for lista in grupo['sintomas'] for s in lista})

    if len(todos_sintomas) == 0:
        continue

    amostras = gerar_amostras(disease, todos_sintomas, n_amostras=100)

    for texto in amostras:
        registros.append({'diseases': disease, 'texto': texto})

df_gold_enriquecido = pd.DataFrame(registros)

print(f"Total de amostras geradas: {len(df_gold_enriquecido)}")
print(f"\nAmostras por doença:")
print(df_gold_enriquecido.groupby('diseases').size().describe())

abaixo = df_gold_enriquecido.groupby('diseases').size()
print(f"\nDoenças abaixo de 100 amostras: {(abaixo < 100).sum()}")
if (abaixo < 100).sum() > 0:
    print(abaixo[abaixo < 100])

df_gold = df_gold_enriquecido.copy()

df_gold.to_csv("../gold/Gold.csv", index=False)

Total de amostras geradas: 75535

Amostras por doença:
count    773.000000
mean      97.716688
std       12.508803
min       15.000000
25%      100.000000
50%      100.000000
75%      100.000000
max      100.000000
dtype: float64

Doenças abaixo de 100 amostras: 25
diseases
adrenal cancer              30
aspergillosis               30
carcinoid syndrome          30
cryptococcosis              30
cushing syndrome            30
diabetes                    30
diabetes insipidus          30
diabetic kidney disease     30
foreign body in the nose    15
hashimoto thyroiditis       30
heart contusion             30
heat stroke                 30
huntington disease          30
insulin overdose            30
kaposi sarcoma              30
moyamoya disease            30
open wound due to trauma    30
open wound of the jaw       30
pulmonic valve disease      30
rheumatic fever             30
thalassemia                 30
toxoplasmosis               30
trichinosis                 30
tricuspid va